In [1]:
# Libraries
import numpy as np 
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform, transform_bounds # Reprojection
from rasterio.transform import Affine
np.set_printoptions(suppress = True) # Turn off scientific notation
from rio_cogeo.cogeo import cog_translate
from rio_cogeo.profiles import cog_profiles
from rio_cogeo import cog_validate, cog_info

In [3]:
# Only run initially!!

#from zipfile import ZipFile

#impervious_zip = './data/percent_impervious/Annual_NLCD_FctImp_2024_CU_C1V1.zip'
#impervious_out = r'./data/percent_impervious'

#with ZipFile(impervious_zip, 'r') as zObject:
    # Extract downloaded percent impervious data and store in data > percent_impervious folder
#    zObject.extractall(path = impervious_out)

In [2]:
# Veiw raw data

impervious_raw = './data/percent_impervious/Annual_NLCD_FctImp_2024_CU_C1V1.tif'

with rasterio.open(impervious_raw, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'uint8', 'nodata': 250.0, 'width': 160000, 'height': 105000, 'count': 1, 'crs': CRS.from_wkt('PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]'), 'transform': Affine(30.0, 0.0, -2415585.0,
       0.0, -30.0, 3314805.0), 'blockxsize': 512, 'blockysize': 512, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: 250.0
CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,

In [2]:
# Update profile 
    # dtype = int8 (int16 used as intermediary)
    # nodata = -10 (& replace nodata cells with new nodata value)

impervious_raw = './data/percent_impervious/Annual_NLCD_FctImp_2024_CU_C1V1.tif'
impervious_profile_update = './data/percent_impervious/impervious_profile_update.tif'

with rasterio.open(impervious_raw) as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.int8,
                   nodata = -10)

    with rasterio.open(impervious_profile_update, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.int16) # int16 as intermediary 
            
            # Replace 250 cells (previous nodata value) to -10 (new nodata value)
            data[data == 250] = -10
            
            # Write out new raster
            dst.write(data.astype(rasterio.int8), 1, window = window)

In [3]:
# Verify profile update

impervious_profile_update = './data/percent_impervious/impervious_profile_update.tif'

with rasterio.open(impervious_profile_update, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 160000, 'height': 105000, 'count': 1, 'crs': CRS.from_wkt('PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]'), 'transform': Affine(30.0, 0.0, -2415585.0,
       0.0, -30.0, 3314805.0), 'blockxsize': 512, 'blockysize': 512, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2

In [4]:
# Reproject to epsg:5070; 30m cells 

impervious_profile_update = './data/percent_impervious/impervious_profile_update.tif'
impervious_reprojected = './data/percent_impervious/impervious_reprojected.tif' 

dst_crs = 'epsg:5070' # Target crs
res = 30 # Res in m

with rasterio.open(impervious_profile_update) as src:
    # Bounds for target crs
    dst_bounds = transform_bounds(src.crs, dst_crs, *src.bounds)
    left, bottom, right, top = dst_bounds
    # Cols and rows
    dst_width = int((right - left) / res)
    dst_height = int((top - bottom) / res)

    # Affine transformation for specific res
    dst_transform = Affine(res, 0, left, 0, -res, top)

    # Set output properties
    dst_profile = src.profile.copy()
    dst_profile.update(
        {
        'crs': dst_crs,
        'transform': dst_transform,
        'width': dst_width,
        'height': dst_height,
        'nodata': -10
        }
    )
    
    with rasterio.open(impervious_reprojected, 'w', **dst_profile) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source = rasterio.band(src, i),
                destination = rasterio.band(dst, i),
                src_transform = src.transform,
                src_crs = src.crs,
                dst_transform = dst_transform,
                dst_crs = dst_crs,
                resampling = Resampling.bilinear # Bilinear for numerical data
            )

In [5]:
# Verify reprojection

impervious_reprojected = './data/percent_impervious/impervious_reprojected.tif' 

with rasterio.open(impervious_reprojected, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 160000, 'height': 104999, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2415585.0000220123,
       0.0, -30.0, 3314804.999958794), 'blockxsize': 512, 'blockysize': 512, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CR

In [6]:
# Co-register percent impervious with dist to gwt raster (inches convert)

# Define co-register function - BILINEAR RESAMPLING
def coregister_rasters(infile, match, outfile):
    """Reproject a file to match the shape and projection of existing raster. 
    
    Parameters
    ----------
    infile : (string) path to input file to reproject
    match : (string) path to raster with desired shape and projection 
    outfile : (string) path to output file tif
    """
    # Open input
    with rasterio.open(infile) as src:
        src_transform = src.transform
        
        # Open input to match
        with rasterio.open(match) as match:
            dst_crs = match.crs
            dst_transform = match.transform # Ensures resolutions of outfile and match will be exactly the same
            dst_width = match.width
            dst_height = match.height

        # Set properties for output
        dst_kwargs = src.meta.copy()
        dst_kwargs.update({'crs': dst_crs,
                           'transform': dst_transform,
                           'width': dst_width,
                           'height': dst_height,
                           'nodata': -10})
        print('Coregistered to shape:', dst_height, dst_width,'\n Affine', dst_transform)
        
        # Open output
        with rasterio.open(outfile, "w", **dst_kwargs) as dst:
            # Iterate through bands and write using reproject function
            for i in range(1, src.count + 1):
                reproject(
                    source = rasterio.band(src, i),
                    destination = rasterio.band(dst, i),
                    src_transform = src.transform,
                    src_crs = src.crs,
                    dst_transform = dst_transform,
                    dst_crs = dst_crs,
                    resampling = Resampling.bilinear)
                

# Apply coregister_rasters
impervious_reprojected = './data/percent_impervious/impervious_reprojected.tif' # Input 
ref_raster = './data/SSURGO_raw/dist_GWT/gwt_inches.tif' # Match
impervious_coregisterd = './data/percent_impervious/impervious_coregistered.tif' # Output

coregister_rasters(
    infile = impervious_reprojected,
    match = ref_raster,
    outfile = impervious_coregisterd
)

Coregistered to shape: 96751 153996 
 Affine | 30.00, 0.00,-2356125.00|
| 0.00,-30.00, 3172575.00|
| 0.00, 0.00, 1.00|


In [7]:
# Verify co-registered raster

impervious_coregisterd = './data/percent_impervious/impervious_coregistered.tif' 

with rasterio.open(impervious_coregisterd, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 153996, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolution (30.0, 30.0)
Mi

In [8]:
# Remove "excess" cells 
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif' 
impervious_coregisterd = './data/percent_impervious/impervious_coregistered.tif' 
impervious_MATCH = './data/percent_impervious/impervious_MATCH.tif' 

# Open hsg composite raster - want conus cells to match THIS raster
with rasterio.open(hsg_final_composite) as conus:

    # Nodata value for the conus raster
    conus_nodata = conus.nodata 
    # Profile conus raster
    profile = conus.profile.copy()

    # Open raster - want to convert any cells containing data where conus contains NODATA to nodata
    with rasterio.open(impervious_coregisterd) as src:
        
        src_nodata = src.nodata # Nodata value 
        
        # Open output raster
        with rasterio.open(impervious_MATCH, 'w', **profile) as dst:
        
            for ji, window in conus.block_windows(1):
            
                # hsg composite raster data
                conus_data = conus.read(1, window = window)
            
                # Land cover raster data
                src_data = src.read(1, window = window)
            
                # Identify cells where conus_data == nodata value
                remove_mask = (conus_data == conus_nodata)
            
                # Convert cells in src where conus is nodata to the nodata value
                src_data[remove_mask] = src_nodata
            
                # Write out
                dst.write(src_data, 1, window = window)

In [9]:
# Verify match

impervious_MATCH = './data/percent_impervious/impervious_MATCH.tif' 

with rasterio.open(impervious_MATCH, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolutio

In [4]:
# Apply min-max scaling  

impervious_MATCH = './data/percent_impervious/impervious_MATCH.tif' 
impervious_standardized = './data/percent_impervious/impervious_standardized.tif' 

with rasterio.open(impervious_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.float32)
    
    # Set initial global min and max
    global_min = np.inf # Highest possible number so anythign will automatically be less
    global_max = -np.inf # Lowest possible number so anything will automatically be greater
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Sanity check: make sure min and max are reasonable values
    print(f'Global min: {global_min}')
    print(f'Global max: {global_max}')
    
    with rasterio.open(impervious_standardized, mode = 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window, masked = True)
            
            # Apply min-max scaling
            rescaled = ((data - global_min) / (global_max - global_min) * 10)
            
            # Fill masked (nodata values) with the nodata value
            rescaled_filled = rescaled.filled(src.nodata)
            
            # Write out raster
            dst.write(rescaled_filled.astype(np.float32), 1, window = window)

Global min: 0
Global max: 100


In [5]:
# Verify min-max scaling

impervious_standardized = './data/percent_impervious/impervious_standardized.tif' 

with rasterio.open(impervious_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [2]:
# Save out preprocessed percent impervious as cloud optimized GeoTiff - save to scratch

impervious_coregisterd = './data/percent_impervious/impervious_coregistered.tif' 
cog_out_impervious_raw_viewing = '/scratch/bfqp/cchan2/rasters/impervious_RAW_VIEWING.tif' # SAVE TO SCRATCH

profile = cog_profiles.get('lzw')

profile.update(
    compress = 'DEFLATE',
    predictor = 3, # 2 for integer; 3 for float
    tiled = True,
    blockxsize = 512,
    blockysize = 512, 
    bigtiff = 'YES')

cog_translate(
    impervious_coregisterd, # dont include argument
    cog_out_impervious_raw_viewing, # dont include argument
    profile, # dont include argument
    overview_resampling = 'bilinear', # Change based on data values
    nodata = -10,
    use_cog_driver = True,
    in_memory = False, 
    web_optimized = True)

Reading input: ./data/percent_impervious/impervious_coregistered.tif

Adding overviews...
Updating dataset tags...
Writing output to: /scratch/bfqp/cchan2/rasters/impervious_RAW_VIEWING.tif


In [3]:
# Validate cogeo raster
cog_out_impervious_raw_viewing = '/scratch/bfqp/cchan2/rasters/impervious_RAW_VIEWING.tif' 

cog_validate(cog_out_impervious_raw_viewing)

(True, [], [])

In [4]:
# Copy from scratch back into projects folder
import shutil

cog_out_impervious_raw_viewing = '/scratch/bfqp/cchan2/rasters/impervious_RAW_VIEWING.tif' 
cog_out_impervious_raw_viewing_PROJECTS = '/projects/bfqp/cchan2/data/percent_impervious/impervious_RAW_VIEWING.tif'

shutil.copy(cog_out_impervious_raw_viewing, cog_out_impervious_raw_viewing_PROJECTS)

'/projects/bfqp/cchan2/data/percent_impervious/impervious_RAW_VIEWING.tif'

In [5]:
# Verify processed cogeo raster

cog_out_impervious_raw_viewing_PROJECTS = './data/percent_impervious/impervious_RAW_VIEWING.tif' 

with rasterio.open(cog_out_impervious_raw_viewing_PROJECTS, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 182784, 'height': 107776, 'count': 1, 'crs': CRS.from_wkt('PROJCS["WGS 84 / Pseudo-Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",0],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],EXTENSION["PROJ4","+proj=merc +a=6378137 +b=6378137 +lat_ts=0 +lon_0=0 +x_0=0 +y_0=0 +k=1 +units=m +nadgrids=@null +wktext +no_defs"],AUTHORITY["EPSG","3857"]]'), 'transform': Affine(38.2185141425881, 0.0, -14245416.087451734,
       0.0, -38.2185141425881, 6731350.458905771), 'blockxsize': 512, 'blockysize': 512, 'tiled': True, 'compress': '

In [4]:
# If this runs, this is just a test for rescaling - will be saving the final rasters in another folder!!

# Apply min-max scaling to the raster

impervious2 = './data/percent_impervious/impervious_reprojected.tif' # Input
impervious3 = './data/percent_impervious/impervious_rescaled_final.tif' # Output

with rasterio.open(impervious2, mode = 'r') as src:
    profile = src.profile
          
    # Set initial global min and max
    global_min = np.inf # Highest possible number so anythign will automatically be less
    global_max = -np.inf # Lowest possible number so anything will automatically be greater
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Sanity check: make sure min and max are reasonable values
    print(f'Global min: {global_min}')
    print(f'Global max: {global_max}')
    
    with rasterio.open(impervious3, mode = 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window, masked = True)
            
            # Apply min-max scaling
            rescaled = ((data - global_min) / (global_max - global_min) * 10)
            
            # Fill masked (nodata values) with the nodata value
            rescaled_filled = rescaled.filled(src.nodata)
            
            # Write out raster
            dst.write(rescaled_filled.astype(np.float32), 1, window = window)

Global min: 0.0
Global max: 100.0


In [ ]:
# started at 2:42 PM
# Global min and max at 2:50
# Edned at 3:01

In [5]:
# Verify rescaling

with rasterio.open(impervious3, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Print the min and max rescaled values
    print(f'Rescaled min: {global_min}')
    print(f'Rescaled max: {global_max}')
        

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -99999.0, 'width': 160000, 'height': 105000, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(29.999999999985338, 0.0, -2415585.0000220123,
       0.0, -29.999999999985338, 3314804.999958794), 'blockxsize': 160000, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}
Nod